In [1]:
from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister, ClassicalRegister

# Computational qubits
data = QuantumRegister(3, "data")

# Auxiliary qubits
anc = AncillaRegister(1, "anc")

# Classical measurement register
out = ClassicalRegister(3, "out")

qc = QuantumCircuit(data, anc, out)

In [2]:
# Prepare computational input
qc.x(data[0])
qc.x(data[1])

# Calculate AND into the ancilla
qc.ccx(data[0], data[1], anc[0])

# Use the result
qc.cx(anc[0], data[2])

# Uncompute the ancilla
qc.ccx(data[0], data[1], anc[0])

# Measure computational qubits
qc.measure(data, out)

qc.draw()

┌───┐                  ┌─┐   
data_0: ┤ X ├──■─────────■─────┤M├───
        ├───┤  │         │     └╥┘┌─┐
data_1: ┤ X ├──■─────────■──────╫─┤M├
        └───┘  │  ┌───┐  │  ┌─┐ ║ └╥┘
data_2: ───────┼──┤ X ├──┼──┤M├─╫──╫─
             ┌─┴─┐└─┬─┘┌─┴─┐└╥┘ ║  ║ 
   anc: ─────┤ X ├──■──┤ X ├─╫──╫──╫─
             └───┘     └───┘ ║  ║  ║ 
 out: 3/═════════════════════╩══╩══╩═
                             2  0  1

In [4]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)

isa_circuit = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=4096)

result = job.result()
counts = result[0].data.out.get_counts()

print(counts)

C:\Users\compu\AppData\Local\Temp\ipykernel_11116\111134905.py:15: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


{'111': 3547, '010': 121, '011': 109, '001': 49, '110': 105, '101': 128, '100': 28, '000': 9}
